# PL 健康诊断

分步验证 Overlay 是否能安全加载。在某 cell 挂死时,问题定位在该 cell 之前。

- 步骤 1 挂 → bitstream 下载问题
- 步骤 2 挂 → FCLK 时钟问题
- 步骤 3 挂 → AXI 总线问题
- 步骤 4 挂 → HLS IP 问题
- 全过 → 可运行 bench_matmul.ipynb

In [ ]:
# 步骤 0.5: 检查当前 FCLK 状态 (在加载 overlay 之前)
# UG585: FCLK_CTRL0 (SLCR+0x170) bit24=CLKACT, bit23:21=DIVISOR1,
#         bit20:16=SRCSEL, bit13:8=DIVISOR0
import mmap, os
fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
slcr = mmap.mmap(fd, 0x1000, offset=0xF8000000, prot=mmap.PROT_READ)
fclk = int.from_bytes(slcr[0x170:0x174], 'little')
slcr.close()
os.close(fd)
clkact = (fclk >> 24) & 0x1
print(f'Boot 后 FCLK_CTRL0 = 0x{fclk:08X} (CLKACT bit24={clkact})')
if clkact:
    print('[OK] FCLK_CLK0 在 boot 后已使能 — base overlay 正常')
else:
    print('[WARN] FCLK_CLK0 CLKACT=0 — base overlay 未使能 FCLK0')
    print('  → 可能正常 (base overlay 用 FCLK1/2/3), 继续步骤 1')

## 步骤 1: 下载 bitstream

In [ ]:
# 先试 Pynq Overlay (能处理 .bit + PS7 init)
try:
    from pynq import Overlay
    ol = Overlay('/home/xilinx/pynq/overlays/matmul_systolic/design_1.bit')
    print('[OK] Pynq Overlay 加载成功 (PS7 init 已应用)')
    pynq_ok = True
except Exception as e:
    print(f'[WARN] Pynq Overlay 失败: {e}')
    pynq_ok = False

In [ ]:
# 如果 Pynq 失败, 用 FPGA manager + .bin (需 sudo, Jupyter 通常无权限)
# 这里只做检测, 不强制
if not pynq_ok:
    print('Pynq Overlay 失败, 尝试诊断原因...')
    print('如果这里挂死, 说明 PS7 配置问题 (DDR/MIO/FCLK)')

## 步骤 2: 验证 FCLK_CLK0 时钟

In [ ]:
# 用 Pynq Clocks API (走 kernel clock framework, 不是直接写 SLCR)
# Linux clk-fclk-zynq 驱动 claim 了 FCLK 寄存器, 直接写 SLCR 会被忽略
from pynq import Clocks
import time, mmap, os

# 读当前 FCLK0 频率
try:
    fclk0_mhz = Clocks.fclk0
    print(f'当前 FCLK0 = {fclk0_mhz} MHz')
except Exception as e:
    print(f'读 FCLK0 失败: {e}')
    fclk0_mhz = 0

# 设置 FCLK0 = 100MHz (Pynq 会通过 kernel clock framework 使能)
if fclk0_mhz != 100:
    print('设置 FCLK0 = 100 MHz ...')
    try:
        Clocks.fclk0 = 100
        time.sleep(0.2)
        fclk0_new = Clocks.fclk0
        print(f'  设置后 FCLK0 = {fclk0_new} MHz')
    except Exception as e:
        print(f'  [FAIL] 设置失败: {e}')
else:
    print('[OK] FCLK0 已是 100 MHz')

# 验证 SLCR 寄存器 — 修正字段位置 (UG585)
# FCLK_CTRL0 (0x170):
#   bit 24     CLKACT  (不是 bit 0!)
#   bit 23:21  DIVISOR1
#   bit 20:16  SRCSEL
#   bit 13:8   DIVISOR0
#   bit 0      CLKACT1 (第二个时钟输出, 通常不用)
fd = os.open('/dev/mem', os.O_RDONLY | os.O_SYNC)
slcr = mmap.mmap(fd, 0x1000, offset=0xF8000000, prot=mmap.PROT_READ)
fclk = int.from_bytes(slcr[0x170:0x174], 'little')
slcr.close(); os.close(fd)
clkact24 = (fclk >> 24) & 0x1
div0 = (fclk >> 8) & 0x3F
div1 = (fclk >> 21) & 0x7
srcsel = (fclk >> 16) & 0x7
print(f'验证: FCLK_CTRL0 = 0x{fclk:08X}')
print(f'  CLKACT(bit24)={clkact24}, SRCSEL={srcsel}, DIV0={div0}, DIV1={div1}')
if clkact24:
    print('[OK] FCLK_CLK0 已使能 (CLKACT bit24=1)')
else:
    print('[FAIL] CLKACT bit24=0, FCLK 未真正使能')

## 步骤 3: 读 DMA 寄存器 (验证 AXI 总线)

In [ ]:
# 读 DMA 寄存器 (验证 AXI 总线 + PL 时钟是否真的活着)
# 即使 SLCR CLKACT=0, 如果 Pynq Clocks API 通过另一个路径使能了时钟,
# DMA 寄存器应该可读
from pynq import MMIO
try:
    dma_a = MMIO(0x40400000, 0x10000)
    mm2s_sr = dma_a.read(0x04)   # MM2S_DMASR
    print(f'DMA_A MM2S_DMASR = 0x{mm2s_sr:08X}')
    if mm2s_sr & 0x1:
        print('[OK] DMA_A 可读 (Halted=1) → PL 时钟活着!')
    else:
        print(f'[WARN] DMA_A Halted=0 (0x{mm2s_sr:08X})')
except Exception as e:
    print(f'[FAIL] DMA 读取失败: {e}')
    print('  → PL 时钟未起, AXI 总线无响应')

## 步骤 4: 读 matmul_top 控制寄存器 (验证 HLS IP)

In [ ]:
# matmul_top_0 基地址 0x43C00000
ip = MMIO(0x43C00000, 0x10000)
ap_ctrl = ip.read(0x00)
m_val = ip.read(0x10)
n_val = ip.read(0x18)
k_val = ip.read(0x20)
print(f'matmul_top AP_CTRL = 0x{ap_ctrl:08X}')
print(f'  M={m_val}, N={n_val}, K={k_val} (应全为 0)')
if ap_ctrl & 0x4:
    print('[OK] ap_idle=1, IP 空闲健康')
else:
    print(f'[WARN] ap_idle=0 (0x{ap_ctrl:08X})')

## 全部通过

In [ ]:
print('=' * 50)
print('  诊断完成! 如果上面都 OK, 可运行 bench_matmul.ipynb')
print('=' * 50)